In [1]:
import math

def minimax(curDepth, nodeIndex, maxTurn, scores, targetDepth, counter=None):
    if curDepth == targetDepth:                       # leaf = terminal utility
        if counter is not None: counter[0] += 1
        return scores[nodeIndex]
    if maxTurn:
        return max(
            minimax(curDepth + 1, nodeIndex * 2, False, scores, targetDepth, counter),
            minimax(curDepth + 1, nodeIndex * 2 + 1, False, scores, targetDepth, counter))
    else:
        return min(
            minimax(curDepth + 1, nodeIndex * 2, True, scores, targetDepth, counter),
            minimax(curDepth + 1, nodeIndex * 2 + 1, True, scores, targetDepth, counter))

def minimax_counted(scores):
    depth = int(math.log2(len(scores)))
    c = [0]
    v = minimax(0, 0, True, scores, depth, c)
    return v, c[0]

def minimax_move(scores):
    
    depth = int(math.log2(len(scores)))
    vals = [minimax(1, i, False, scores, depth) for i in (0, 1)]
    best = 0 if vals[0] >= vals[1] else 1             # strict preference for the first on ties
    return best, vals

def level_table(scores):
    
    depth = int(math.log2(len(scores)))
    rows = []
    for d in range(depth, -1, -1):
        kind = "LEAF" if d == depth else ("MAX" if d % 2 == 0 else "MIN")
        vals = [minimax(d, i, d % 2 == 0, scores, depth) for i in range(2 ** d)]
        rows.append((d, kind, vals))
    return rows

if __name__ == "__main__":
    # Section 5.1 supplied tree, padded exactly as the manual's driver does
    scores = [3, 5, 2, 9, 0, 7]
    tree_depth = math.ceil(math.log2(len(scores)))
    scores += [0] * (2 ** tree_depth - len(scores))   # PADDING ASSUMPTION: dummy leaves = 0
    print("Supplied tree (padded):", scores, "depth", tree_depth)
    print("\n--- Bottom-up hand trace (compare with your paper trace) ---")
    for d, kind, vals in level_table(scores):
        print(f"depth {d} {kind:4s}:", vals)
    v, n = minimax_counted(scores)
    m, child_vals = minimax_move(scores)
    print("\nroot value :", v)
    print("root move  :", "LEFT" if m == 0 else "RIGHT", "| child (MIN) values", child_vals)
    print("leaves evaluated by minimax:", n)
    # manual driver output
    print("manual driver prints: The optimal value is:", minimax(0, 0, True, scores, tree_depth))

    print("\n--- Section 5.2 tree ---")
    s2 = [3, 5, 2, 9, 3, 5, 2, 9]
    for d, kind, vals in level_table(s2):
        print(f"depth {d} {kind:4s}:", vals)
    v2, n2 = minimax_counted(s2)
    print("root value:", v2, "| leaves evaluated:", n2, "| move:", "LEFT" if minimax_move(s2)[0] == 0 else "RIGHT")

Supplied tree (padded): [3, 5, 2, 9, 0, 7, 0, 0] depth 3

--- Bottom-up hand trace (compare with your paper trace) ---
depth 3 LEAF: [3, 5, 2, 9, 0, 7, 0, 0]
depth 2 MAX : [5, 9, 7, 0]
depth 1 MIN : [5, 0]
depth 0 MAX : [5]

root value : 5
root move  : LEFT | child (MIN) values [5, 0]
leaves evaluated by minimax: 8
manual driver prints: The optimal value is: 5

--- Section 5.2 tree ---
depth 3 LEAF: [3, 5, 2, 9, 3, 5, 2, 9]
depth 2 MAX : [5, 9, 5, 9]
depth 1 MIN : [5, 5]
depth 0 MAX : [5]
root value: 5 | leaves evaluated: 8 | move: LEFT


In [3]:
def alphabeta(depth, index, maxTurn, scores, target, alpha, beta, counter=None, visited=None):
    if visited is not None: visited.append((depth, index))
    if depth == target:
        if counter is not None: counter[0] += 1
        return scores[index]
    if maxTurn:
        value = float('-inf')
        for i in range(2):
            value = max(value, alphabeta(depth+1, index*2+i, False, scores, target, alpha, beta, counter, visited))
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value
    else:
        value = float('inf')
        for i in range(2):
            value = min(value, alphabeta(depth+1, index*2+i, True, scores, target, alpha, beta, counter, visited))
            beta = min(beta, value)
            if alpha >= beta:
                break
        return value

def alphabeta_counted(scores):
    depth = int(math.log2(len(scores)))
    c = [0]
    v = alphabeta(0, 0, True, scores, depth, float('-inf'), float('inf'), c)
    return v, c[0]

def alphabeta_move(scores):
    """Root decision with alpha-beta. Strict '>' keeps the first best child on ties."""
    depth = int(math.log2(len(scores)))
    alpha, best, c = float('-inf'), 0, [0]
    for i in (0, 1):
        v = alphabeta(1, i, False, scores, depth, alpha, float('inf'), c)
        if v > alpha:
            alpha, best = v, i
    return best, alpha, c[0]

def alphabeta_traced(scores):
    """Same logic as alphabeta(), printing every visit, bound update and prune."""
    target = int(math.log2(len(scores)))
    leaves = [0]
    def go(d, i, is_max, alpha, beta):
        pad = "  " * d
        if d == target:
            leaves[0] += 1
            print(f"{pad}leaf ({d},{i}) = {scores[i]}   [leaf #{leaves[0]}]")
            return scores[i]
        print(f"{pad}enter ({d},{i}) {'MAX' if is_max else 'MIN'}  alpha={alpha} beta={beta}")
        value = float('-inf') if is_max else float('inf')
        for k in range(2):
            child = go(d+1, i*2+k, not is_max, alpha, beta)
            if is_max:
                value = max(value, child); alpha = max(alpha, value)
            else:
                value = min(value, child); beta = min(beta, value)
            print(f"{pad}  after child {k}: value={value} alpha={alpha} beta={beta}")
            if alpha >= beta:
                if k == 0:
                    print(f"{pad}  PRUNE: alpha>=beta, skip child (({d+1},{i*2+1}) and its subtree)")
                break
        return value
    v = go(0, 0, True, float('-inf'), float('inf'))
    return v, leaves[0]

if __name__ == "__main__":
    s2 = [3, 5, 2, 9, 3, 5, 2, 9]
    print("=== Section 5.2 tree", s2, "===")
    v, n = alphabeta_traced(s2)
    print(f"\nalpha-beta value {v}, leaves evaluated {n} of {len(s2)}")
    mm_v, mm_n = minimax_counted(s2)
    print(f"minimax    value {mm_v}, leaves evaluated {mm_n} of {len(s2)}")
    print("root move (minimax / alpha-beta):", minimax_move(s2)[0], "/", alphabeta_move(s2)[0], "(0=LEFT, 1=RIGHT)")
    assert v == mm_v and n <= mm_n
    print("\n=== Section 5.1 supplied tree (padded) ===")
    s1 = [3, 5, 2, 9, 0, 7, 0, 0]
    v, n = alphabeta_traced(s1)
    print(f"\nalpha-beta value {v}, leaves evaluated {n} of {len(s1)}  (no pruning for THIS ordering)")
    print("minimax    value", minimax_counted(s1)[0], "leaves", minimax_counted(s1)[1])

=== Section 5.2 tree [3, 5, 2, 9, 3, 5, 2, 9] ===
enter (0,0) MAX  alpha=-inf beta=inf
  enter (1,0) MIN  alpha=-inf beta=inf
    enter (2,0) MAX  alpha=-inf beta=inf
      leaf (3,0) = 3   [leaf #1]
      after child 0: value=3 alpha=3 beta=inf
      leaf (3,1) = 5   [leaf #2]
      after child 1: value=5 alpha=5 beta=inf
    after child 0: value=5 alpha=-inf beta=5
    enter (2,1) MAX  alpha=-inf beta=5
      leaf (3,2) = 2   [leaf #3]
      after child 0: value=2 alpha=2 beta=5
      leaf (3,3) = 9   [leaf #4]
      after child 1: value=9 alpha=9 beta=5
    after child 1: value=5 alpha=-inf beta=5
  after child 0: value=5 alpha=5 beta=inf
  enter (1,1) MIN  alpha=5 beta=inf
    enter (2,2) MAX  alpha=5 beta=inf
      leaf (3,4) = 3   [leaf #5]
      after child 0: value=3 alpha=5 beta=inf
      leaf (3,5) = 5   [leaf #6]
      after child 1: value=5 alpha=5 beta=inf
    after child 0: value=5 alpha=5 beta=5
    PRUNE: alpha>=beta, skip child ((2,3) and its subtree)
  after child 1: 

In [4]:
import itertools, statistics
from collections import Counter

def nest(scores):
    if len(scores) == 1: return scores[0]
    h = len(scores) // 2
    return [nest(scores[:h]), nest(scores[h:])]

def flatten(t):
    return [t] if not isinstance(t, list) else flatten(t[0]) + flatten(t[1])

def value(t, is_max):
    if not isinstance(t, list): return t
    a, b = value(t[0], not is_max), value(t[1], not is_max)
    return max(a, b) if is_max else min(a, b)

def swap_by_mask(t, bits):
    """Swap children at internal nodes (pre-order) wherever the next bit is 1."""
    if not isinstance(t, list): return t
    flip = next(bits)
    l, r = swap_by_mask(t[0], bits), swap_by_mask(t[1], bits)
    return [r, l] if flip else [l, r]

def order_by_value(t, is_max, best_first):
    """best_first: MAX node -> larger child first, MIN node -> smaller child first."""
    if not isinstance(t, list): return t
    l = order_by_value(t[0], not is_max, best_first)
    r = order_by_value(t[1], not is_max, best_first)
    vl, vr = value(l, not is_max), value(r, not is_max)
    first_is_l = (vl >= vr) if is_max else (vl <= vr)
    if not best_first: first_is_l = not first_is_l
    return [l, r] if first_is_l else [r, l]

def report(label, scores):
    mv, mn = minimax_counted(scores)
    av, an = alphabeta_counted(scores)
    mm, ab = minimax_move(scores)[0], alphabeta_move(scores)[0]
    assert mv == av and an <= mn and mm == ab
    print(f"{label:22s} {str(scores):28s} minimax {mv}/{mn} leaves | alpha-beta {av}/{an} leaves | "
          f"saved {mn-an} | chosen position {'LEFT' if ab == 0 else 'RIGHT'}")
    return an

def exhaustive(name, scores):
    t = nest(scores); dist = Counter(); vals = set(); moves_ok = True
    for mask in itertools.product([0, 1], repeat=7):
        s = flatten(swap_by_mask(t, iter(mask)))
        mv, mn = minimax_counted(s); av, an = alphabeta_counted(s)
        vals.add((mv, av)); dist[an] += 1
        assert mv == av and an <= mn
    all_counts = [k for k, c in dist.items() for _ in range(c)]
    print(f"\n--- exhaustive: all 128 sibling orderings of {name} {scores} ---")
    print("root value (minimax, alpha-beta) pairs seen:", vals)
    print("alpha-beta leaves -> number of orderings:", dict(sorted(dist.items())))
    print(f"min {min(all_counts)}  max {max(all_counts)}  mean {statistics.mean(all_counts):.2f}  (minimax always 8)")

if __name__ == "__main__":
    for name, sc in (("Section 5.2 tree", [3, 5, 2, 9, 3, 5, 2, 9]),
                     ("Section 5.1 tree (padded)", [3, 5, 2, 9, 0, 7, 0, 0])):
        print(f"\n=== {name} ===")
        t = nest(sc)
        report("original", sc)
        report("mirror (all swapped)", flatten(swap_by_mask(t, iter([1]*7))))
        report("swap root only", flatten(swap_by_mask(t, iter([1] + [0]*6))))
        report("best-first (ideal)", flatten(order_by_value(t, True, True)))
        report("worst-first", flatten(order_by_value(t, True, False)))
        exhaustive(name, sc)


=== Section 5.2 tree ===
original               [3, 5, 2, 9, 3, 5, 2, 9]     minimax 5/8 leaves | alpha-beta 5/6 leaves | saved 2 | chosen position LEFT
mirror (all swapped)   [9, 2, 5, 3, 9, 2, 5, 3]     minimax 5/8 leaves | alpha-beta 5/8 leaves | saved 0 | chosen position LEFT
swap root only         [3, 5, 2, 9, 3, 5, 2, 9]     minimax 5/8 leaves | alpha-beta 5/6 leaves | saved 2 | chosen position LEFT
best-first (ideal)     [5, 3, 9, 2, 5, 3, 9, 2]     minimax 5/8 leaves | alpha-beta 5/5 leaves | saved 3 | chosen position LEFT
worst-first            [2, 9, 3, 5, 2, 9, 3, 5]     minimax 5/8 leaves | alpha-beta 5/8 leaves | saved 0 | chosen position LEFT

--- exhaustive: all 128 sibling orderings of Section 5.2 tree [3, 5, 2, 9, 3, 5, 2, 9] ---
root value (minimax, alpha-beta) pairs seen: {(5, 5)}
alpha-beta leaves -> number of orderings: {5: 16, 6: 48, 7: 16, 8: 48}
min 5  max 8  mean 6.75  (minimax always 8)

=== Section 5.1 tree (padded) ===
original               [3, 5, 2, 9, 0,

In [6]:
"""Lab 5 - edge cases and randomized equivalence check.
Run: python lab5_check.py
"""
import random

print("=== Test 1: equal leaves (ties) ===")
s = [4] * 8
print("minimax", minimax_counted(s), "| alpha-beta", alphabeta_counted(s),
      "| moves", minimax_move(s)[0], alphabeta_move(s)[0])

print("\n=== Test 2: negative utilities ===")
s = [-3, -5, -2, -9, -3, -5, -2, -9]
print("minimax", minimax_counted(s), "| alpha-beta", alphabeta_counted(s))

print("\n=== Test 3: padding is an ASSUMPTION (manual 5.1 tree) ===")
base = [3, 5, 2, 9, 0, 7]
for pad in (0, 8):
    s = base + [pad, pad]
    print(f"pad leaves = {pad}: tree {s} -> minimax value {minimax_counted(s)[0]}, "
          f"alpha-beta value {alphabeta_counted(s)[0]}")
print("-> the root value changes with the invented padding, so padding must be documented.")

print("\n=== Test 4: minimal tree (depth 1, two leaves) ===")
print("minimax", minimax_counted([6, 2]), "| alpha-beta", alphabeta_counted([6, 2]))

print("\n=== Test 5: randomized equivalence (2000 random depth-4 trees, 16 leaves) ===")
random.seed(42)
total_mm = total_ab = pruned_trees = 0
for _ in range(2000):
    s = [random.randint(-10, 10) for _ in range(16)]
    mv, mn = minimax_counted(s); av, an = alphabeta_counted(s)
    assert mv == av, ("value mismatch", s)
    assert an <= mn, ("alpha-beta evaluated more leaves", s)
    assert minimax_move(s)[0] == alphabeta_move(s)[0], ("move mismatch", s)
    total_mm += mn; total_ab += an; pruned_trees += an < mn
print("all 2000 trees: same root value, same root move, alpha-beta leaves <= minimax leaves")
print(f"average leaves: minimax {total_mm/2000:.2f}, alpha-beta {total_ab/2000:.2f}; "
      f"trees with any pruning: {pruned_trees}/2000")

=== Test 1: equal leaves (ties) ===
minimax (4, 8) | alpha-beta (4, 5) | moves 0 0

=== Test 2: negative utilities ===
minimax (-3, 8) | alpha-beta (-3, 5)

=== Test 3: padding is an ASSUMPTION (manual 5.1 tree) ===
pad leaves = 0: tree [3, 5, 2, 9, 0, 7, 0, 0] -> minimax value 5, alpha-beta value 5
pad leaves = 8: tree [3, 5, 2, 9, 0, 7, 8, 8] -> minimax value 7, alpha-beta value 7
-> the root value changes with the invented padding, so padding must be documented.

=== Test 4: minimal tree (depth 1, two leaves) ===
minimax (6, 2) | alpha-beta (6, 2)

=== Test 5: randomized equivalence (2000 random depth-4 trees, 16 leaves) ===
all 2000 trees: same root value, same root move, alpha-beta leaves <= minimax leaves
average leaves: minimax 16.00, alpha-beta 11.74; trees with any pruning: 1955/2000


In [7]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def draw(scores, title, fname, visited=None, show_values=True):
    depth = int(math.log2(len(scores)))
    pos = {}
    for i in range(len(scores)): pos[(depth, i)] = (i, 0)
    for d in range(depth - 1, -1, -1):
        for i in range(2 ** d):
            a, b = pos[(d + 1, 2 * i)], pos[(d + 1, 2 * i + 1)]
            pos[(d, i)] = ((a[0] + b[0]) / 2, (depth - d))
    fig, ax = plt.subplots(figsize=(9, 5.2))
    seen = set(visited) if visited is not None else None
    for (d, i), (x, y) in pos.items():
        if d == depth: continue
        for k in (0, 1):
            c = (d + 1, 2 * i + k); cx, cy = pos[c]
            pruned = seen is not None and c not in seen
            ax.plot([x, cx], [y, cy], color="#bbbbbb" if pruned else "#333333",
                    ls="--" if pruned else "-", lw=1.4, zorder=1)
    for (d, i), (x, y) in pos.items():
        pruned = seen is not None and (d, i) not in seen
        is_max = d % 2 == 0
        val = minimax(d, i, is_max, scores, depth)
        marker = "s" if d == depth else ("^" if is_max else "v")
        color = "#e6e6e6" if pruned else ("#fff2cc" if d == depth else ("#d9e2f3" if is_max else "#fce4d6"))
        ax.scatter([x], [y], s=1500 if d < depth else 900, marker=marker, color=color,
                   edgecolors="#bbbbbb" if pruned else "#333333", zorder=2)
        label = str(val) if (show_values and not pruned) else ("X" if pruned else str(val))
        ax.text(x, y - (0.0 if d == depth else -0.03), label, ha="center", va="center",
                fontsize=12, color="#999999" if pruned else "black", zorder=3, fontweight="bold")
    for d in range(depth + 1):
        ax.text(-0.9, depth - d, ("Leaves" if d == depth else ("MAX" if d % 2 == 0 else "MIN")) + f" (depth {d})",
                ha="right", va="center", fontsize=10)
    ax.set_xlim(-3.2, len(scores) - 0.3); ax.set_ylim(-0.6, depth + 0.6)
    ax.axis("off"); ax.set_title(title, fontsize=12)
    fig.tight_layout(); fig.savefig(fname, dpi=130); plt.close(fig)

if __name__ == "__main__":
    s1 = [3, 5, 2, 9, 0, 7, 0, 0]
    draw(s1, "Section 5.1 tree (padded with 0s): Minimax values bottom-up, root = 5, move = LEFT", "tree_minimax.png")
    s2 = [3, 5, 2, 9, 3, 5, 2, 9]
    vis = []
    alphabeta(0, 0, True, s2, 3, float('-inf'), float('inf'), None, vis)
    draw(s2, "Section 5.2 tree: alpha-beta (grey X = pruned, 2 of 8 leaves skipped), root = 5", "tree_alphabeta.png", visited=vis)
    print("saved tree_minimax.png, tree_alphabeta.png")

saved tree_minimax.png, tree_alphabeta.png
